# Throwing Data Away, Carefully: What CIFAR-10 Coreset Experiments Taught Us

*How much of your training set can you drop, and how should you choose what stays? We trained
the same ResNet-18 on subsets picked six different ways, across budgets from 1% to 90%. The
best strategy turned out to depend heavily on how much data you keep.*

## The problem: not all data is worth keeping

Every image you train on costs something to label, store, and compute over — and in most
datasets a large share of examples are near-duplicates or easy cases that stop teaching the
model anything new once it has seen a few of their neighbors. **Coreset selection** keeps the
examples that carry the most signal and drops the rest.

3LC builds a **traversal index** over a model's embeddings: a *farthest-first* ordering that
starts from the most extreme point and repeatedly adds whichever point is farthest from
everything picked so far. Early positions cover the diverse edges of the data; later ones fill
in the crowded middle. Keep the first *b%* of that ordering and you have a diversity-maximizing
subset at budget *b*.

The open question is *how* to build that ordering — and we tested three choices head-to-head.

## The experiment in one minute

- **Task:** CIFAR-10 classification (50k train / 10k val, 10 balanced classes).
- **Recipe:** ResNet-18, 10 epochs, identical for every run — only the *training subset*
  changes, so accuracy differences reflect **which data was selected**.
- **Budgets:** 1%, 5%, 10%, 20%, … , 90%, plus the 100% full-data reference.
- **Score:** best validation accuracy over the 10 epochs.
- **Three design choices (six strategies total):**
  - **Global vs. per-category** — traverse the whole dataset, or separately *within each class*
    (forcing an equal share per class).
  - **Unweighted vs. uncertainty-weighted** — plain diversity, or scale each embedding by
    $(1 - \text{confidence})$ first, tilting selection toward hard examples.
  - **Original vs. reduced-space** — traverse the original high-dimensional embeddings, or
    their 3-D `PaCMAP` / `UMAP` projections.

That's 6 strategies × 11 budgets = **66 training runs**. The full-data model peaks at
**~94.7%**, and three identical full-data runs spanned ~0.3 points — so **treat gaps under
~0.3 points as noise**.

## Results at a glance

Peak validation accuracy (%) for every strategy and budget. **Bold** marks the best strategy
at each budget; the full-data (50k / 100%) reference is **~94.7%**.

| Budget | Original | Original weighted | Per-category | Per-cat weighted | PaCMAP | UMAP |
|:--|--:|--:|--:|--:|--:|--:|
| 90% (45k)  | 94.81 | **94.94** | 94.76 | 94.81 | 94.80 | 94.72 |
| 80% (40k)  | 94.62 | 94.74 | 94.45 | **94.93** | 94.46 | 94.49 |
| 70% (35k)  | 94.50 | 94.64 | 94.25 | **94.86** | 94.66 | 94.34 |
| 60% (30k)  | 94.48 | **94.64** | 94.10 | 94.48 | 93.91 | 93.79 |
| 50% (25k)  | 94.09 | **94.46** | 93.61 | 94.38 | 93.79 | 93.30 |
| 40% (20k)  | 93.59 | **94.11** | 93.67 | 93.63 | 93.33 | 92.93 |
| 30% (15k)  | 93.05 | **93.50** | 92.34 | 92.32 | 92.34 | 92.47 |
| 20% (10k)  | **91.53** | 89.00 | **91.53** | 88.74 | 91.14 | 91.38 |
| 10% (5k)   | 88.75 | 73.10 | **89.27** | 80.34 | 88.00 | 88.60 |
| 5% (2.5k)  | **84.08** | 55.80 | 83.37 | 68.38 | 82.99 | 83.11 |
| 1% (500)   | 52.31 | 23.53 | 55.77 | 40.07 | 53.69 | **62.09** |


## Finding 1 — Original embeddings usually beat the reduced view

**Should the traversal run on the model's original high-dimensional embeddings, or on the
cheaper 3-D UMAP/PaCMAP projection that also powers 3LC's interactive view?**

Across the everyday range (10–90%), the **original** embeddings are the strongest
diversity-only choice, with the widest lead in the middle — around 40–60% they beat UMAP by
~0.7–1.2 points. Compressing embeddings to three dimensions blurs the distances the traversal
relies on, so a subset chosen there covers the real data manifold less faithfully.

![](figures/finding1.png)

At the extreme low end it flips: at 1% (~500 images) **UMAP wins outright** (~62% vs. ~52%).
With only a few hundred picks, farthest-first on raw embeddings wastes them chasing outliers,
while a projection's smoother structure spreads more robustly.

![](figures/finding1-2.png)

**Takeaway:** select on the original embeddings in the normal range; consider a reduced-space
ordering only on tiny budgets. (Projections stay ideal for *visualizing* your data either way.)

## Finding 2 — Chasing hard examples helps later, hurts early

**Does tilting selection toward low-confidence ("hard") examples — scaling each embedding by
`1 − confidence` before the traversal — beat plain diversity?**

It has two personalities, split cleanly around **~30% of the data**:

- **Above ~30%, weighting wins** — about +0.4–0.5 points in the 40–60% range, shrinking as you
  approach the full-data ceiling. Once coverage is handled, hard, near-boundary examples
  sharpen the decision boundary more than another easy duplicate would.
- **Below ~30%, it backfires** — ~16 points behind plain diversity at 10%, ~29 behind at 1%.
  On a tiny budget it spends everything on ambiguous (sometimes mislabeled) images and skips
  the representative examples each class needs; its learning curve even peaks early and then
  slides back down — the signature of overfitting to a small, hard set.

![](figures/finding2.png)

**Takeaway:** uncertainty weighting is a high-budget booster and a low-budget trap — the
familiar active-learning story where coverage wins small and uncertainty wins big.

## Finding 3 — Per-category selection rescues small budgets

Does building the traversal *within each class* — guaranteeing every class an equal share —
beat one global traversal over everything?

- **On tight budgets, class balance helps.** Among diversity-only strategies, per-category is
  best at 10% (~89.3%) and ties the global original at 20%. A global traversal can starve a
  class when only a few hundred points are kept; per-category can't.
- **In the mid-range, global edges ahead.** Around 30–60%, the rigid equal split trails by a
  few tenths — with enough budget, letting diversity allocate (and preserving cross-class
  boundaries) beats strict balance.
- **It defuses the weighting trap.** Adding per-category balance lifts the *weighted* subset by
  ~7 points at 10%, ~13 at 5%, and ~16 at 1% — turning a catastrophe into merely a bad idea.

![](figures/finding3.png)

**Takeaway:** per-category balancing is cheap insurance for coverage — most valuable where the
global methods are weakest (small budgets), and competitive at the high end when paired with
weighting.

## So which should you use? It depends on your budget

There's no single winner — the best strategy rotates with how much data you keep, so you can
choose deliberately:

| Data budget | Good default | Why |
|:--|:--|:--|
| **Tiny: 500 (~1%)** | Reduced-space (**UMAP**) | raw farthest-first wastes a tiny budget on outliers |
| **Small: 2.5-10k (~5–20%)** | Plain diversity, ideally **per-category** | coverage is everything; class balance is cheap insurance; don't weight yet |
| **Medium: 15-30k (~30–60%)** | **Original weighted** | coverage is handled, so hard examples pay off most here |
| **Large: 25k-50k (~70–90%)** | Weighting, and try **per-cat weighted** | still helps, but gains shrink near the full-data ceiling |

![](figures/overview.png)

Two framing points worth keeping in mind:

- **Data efficiency beats decimal points.** Near the top, every method is within a point of
  full data, so the sharper question is *how little data reaches a target*. To hit 94%: global
  weighted needs ~38% of the data, per-cat weighted ~45%, plain original ~48%, and UMAP ~64%.
- **You can beat "all the data."** Several strategies at 80–90% match or exceed the full-data
  run — dropping the least useful ~10–20% (easy, redundant, or noisy images) can be free or
  better.

## Important notes

- **This is CIFAR-10 classification.** The *shape* of these findings should generalize, but the
  exact numbers and cutoffs won't transfer unchanged to your dataset, model, or task — re-run
  the sweep on your own data before committing.
- **A generous confidence signal.** Weighting used confidence from a full-data model; a real
  active-learning loop recomputes it from a weaker in-progress model, which usually helps
  weighting less.

## The bottom line

Coreset selection isn't one method — it's a few levers (where to traverse, whether to weight by
uncertainty, whether to balance by class) whose right settings shift with your data budget.
Match the lever to the budget and you can train on a fraction of the data with little or no
accuracy cost.